# Stage 1 Fine-Tuning on Colab - Tshivenda (ven)

Bootstrap notebook: run top to bottom on a **fresh Colab GPU runtime** and it takes care
of everything - clone, dependencies, data regeneration, training with Drive-backed
checkpoints (so free-tier disconnects RESUME instead of restarting), and final evaluation.

**Before you start (one-time):**
1. Runtime -> Change runtime type -> **GPU** (T4 is fine).
2. You need a Hugging Face account + access token (huggingface.co/settings/tokens, read scope).
3. The ANV/Swivuriso dataset is **gated**: while logged in on the HF website, open
   https://huggingface.co/datasets/dsfsi-anv/za-african-next-voices-compressed and accept
   the terms (approval is automatic). Without this, preprocessing fails on the ANV step.
4. This clones the `feature/tshivenda` branch - it must be pushed to GitHub first.

**Teammates (Setswana/Sepedi)**: clone this pattern - the things to change are the
language code in the preprocessing scripts, the tokenizer dir, and the CSV paths.
NOTE for Sepedi: ANV/Swivuriso has no `nso` config (7-language corpus) - NCHLT only.

## 1. GPU check

In [ ]:
# Should show a T4/V100/A100. If it errors: Runtime -> Change runtime type -> GPU
!nvidia-smi -L

## 2. Clone the repo + install dependencies

No macOS ffmpeg workaround needed here - Colab is Linux and torchcodec works with its system ffmpeg.

In [ ]:
!git clone -b feature/tshivenda https://github.com/Khotso-Bore/MultilingualASR.git
%cd MultilingualASR
# torch is preinstalled on Colab; this adds datasets/transformers/jiwer/etc.
!pip install -q -r requirements.txt

## 3. Hugging Face login (needed for the gated ANV dataset)

In [ ]:
# Paste your HF token when prompted (or store it as a Colab secret named HF_TOKEN
# and use userdata.get). The account must have accepted the ANV gate - see intro.
from huggingface_hub import login
login()

## 4. Mount Google Drive

Checkpoints and results go to Drive so a disconnected session can resume. Needs ~5 GB free Drive space (checkpoints are pruned to the 2 most recent).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/multilingualasr"
OUTPUT_DIR = f"{DRIVE_ROOT}/wav2vec2-ven-stage1"
PREDS_DIR = f"{DRIVE_ROOT}/preds-stage1"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. Regenerate the processed data on this runtime

Downloads from HF and writes ~21 GB under `dataset/` (Colab disk, not Drive). NCHLT ~10 min, ANV ~30-60 min on Colab's connection. Re-runs are needed after every runtime reset - the wavs live on ephemeral disk (only checkpoints persist on Drive).

In [ ]:
!python src/preprocessing/preprocess_nchlt.py
!python src/preprocessing/preprocess_anv.py

## 6. Stage 1 training

Same logic as `finetune_wav2vec2_ven.ipynb` (STAGE=1: NCHLT+ANV combined, ~60k clips), CUDA-adjusted (fp16 on). **Resumes automatically** from the newest Drive checkpoint if the session died mid-run - just re-run the notebook top to bottom.

In [ ]:
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Features, Value
from transformers import (
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer, EarlyStoppingCallback,
)
from transformers.trainer_utils import get_last_checkpoint
from jiwer import wer, cer

DATA = "dataset/processed"
features = Features({"audio": Audio(sampling_rate=16000), "transcript": Value("string")})
dataset_dict = load_dataset("csv", data_files={
    "train": [f"{DATA}/nchlt_ven/train.csv", f"{DATA}/anv_ven/train.csv"],
    "dev": [f"{DATA}/nchlt_ven/validation.csv", f"{DATA}/anv_ven/dev.csv"],
}, features=features)

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("tokenizers/ven")
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def prepare_dataset(batch):
    samples = batch["audio"].get_all_samples()
    array = samples.data.numpy().squeeze()
    batch["input_values"] = processor(array, sampling_rate=16000).input_values[0]
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

dataset_dict = dataset_dict.map(prepare_dataset, remove_columns=dataset_dict["train"].column_names)

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    def __call__(self, feats):
        inputs = [{"input_values": f["input_values"]} for f in feats]
        labels = [{"input_ids": f["labels"]} for f in feats]
        batch = self.processor.pad(inputs, padding=True, return_tensors="pt")
        labels_batch = self.processor.pad(labels=labels, padding=True, return_tensors="pt")
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    ctc_loss_reduction="mean", ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,   # T4 16GB: drop to 4 on OOM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    num_train_epochs=10,
    fp16=True,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    push_to_hub=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorCTCWithPadding(processor=processor),
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# resume from the newest Drive checkpoint if one exists (session died mid-run)
last_ckpt = get_last_checkpoint(OUTPUT_DIR)
print("resuming from:", last_ckpt or "scratch")
trainer.train(resume_from_checkpoint=last_ckpt)

trainer.save_model(f"{OUTPUT_DIR}/final")
processor.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"saved -> {OUTPUT_DIR}/final")

## 7. Evaluate (comparable to the zero-shot baseline table)

Same eval sets, sampling, and normalisation as `src/zero_shot_baseline.py`. Predictions are saved to Drive - feed them to `src/corrupt_transcripts.py --error-model` for the error-propagation study.

In [ ]:
!python src/asr/evaluate_wav2vec2.py --checkpoint {OUTPUT_DIR}/final --save-predictions {PREDS_DIR}